# 02 Feature Engineering

## Mounting Google Drive (if needed)

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# %cd /content/drive/MyDrive/Colab Notebooks/datasci-207-final-project

/content/drive/MyDrive/Colab Notebooks/datasci-207-final-project


## Imports

In [45]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tulipy as ti
import gzip
import shutil

## Data Retrieval

In [41]:
# data = pd.read_csv('SP500_4y_day_data.csv')  # If you had saved it to this file instead
data = pd.read_csv('data/initial_data.csv')
data.head()

,Date_col,Ticker,Open,High,Low,Close,Volume,rank
0,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0
1,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0
2,2021-01-06,A,115.631477,120.687730,115.163842,119.723244,2428500.0,3.0
3,2021-01-07,A,121.126152,124.194981,120.911820,122.908997,1775900.0,4.0
4,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0


In [42]:
data['Date_col'] = pd.to_datetime(data['Date_col'])
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480390 entries, 0 to 480389
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   Date_col  480390 non-null  datetime64[ns]
 1   Ticker    480390 non-null  object        
 2   Open      480390 non-null  float64       
 3   High      480390 non-null  float64       
 4   Low       480390 non-null  float64       
 5   Close     480390 non-null  float64       
 6   Volume    480390 non-null  float64       
 7   rank      480390 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 29.3+ MB


## Feature Engineering

Outcome variables - Close price minus the Open price and variations of that

In [43]:
# Getting difference between close price and open price (outcome variables)
data["Close_Open_Diff"] = data["Close"] - data["Open"]
data["Close_Open_Diff_Perc"] = (data["Close"] - data["Open"]) / data["Open"] * 100

# Getting binary classification for whether a stock ended positive (1) or negative/zero (0)
data["Close_Open_Diff_Binary"] = data["Close_Open_Diff"] > 0

# Getting ternary classification for whether a stock ended positive (2), negative (0), or not much change (1)
close_open_conditions = [
    data['Close_Open_Diff_Perc'] > 0.5,       # Close price was at least 0.5% higher than Open price
    (data['Close_Open_Diff_Perc'] >= -0.5) &  # Close price was between -0.5% and 0.5% of Open price
        (data['Close_Open_Diff_Perc'] <= 0.5),
    data['Close_Open_Diff_Perc'] < -0.5       # Close price was at least 0.5% lower than Open price
]
close_open_choices = [2, 1, 0]  # Defining choices corresponding to conditions
data['Close_Open_Diff_Ternary'] = np.select(close_open_conditions, close_open_choices)  # Applying conditions

data

,Date_col,Ticker,Open,High,Low,Close,Volume,rank,Close_Open_Diff,Close_Open_Diff_Perc,Close_Open_Diff_Binary,Close_Open_Diff_Ternary
0,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0,-0.292272,-0.252231,False,1
1,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0,1.100878,0.953745,True,2
2,2021-01-06,A,115.631477,120.687730,115.163842,119.723244,2428500.0,3.0,4.091767,3.538627,True,2
3,2021-01-07,A,121.126152,124.194981,120.911820,122.908997,1775900.0,4.0,1.782845,1.471891,True,2
4,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0,0.555311,0.450628,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...
480385,2024-12-24,ZTS,163.206597,164.542554,162.249499,164.203583,1023600.0,1001.0,0.996986,0.610874,True,2
480386,2024-12-26,ZTS,163.226540,165.290293,162.528653,165.021118,2167200.0,1002.0,1.794578,1.099440,True,2
480387,2024-12-27,ZTS,164.452833,166.028073,163.027150,164.103897,1800100.0,1003.0,-0.348936,-0.212180,False,1
480388,2024-12-30,ZTS,163.396028,163.555549,160.963380,161.751007,1531400.0,1004.0,-1.645021,-1.006769,False,0


Getting previous day metrics (since we don't know current day's metrics except open) and growth rates

In [44]:
# Getting the first actual trading day
first_trading_date = str(data.loc[data['Date_col'].idxmin(), 'Date_col'].date())

# Getting growth rates and previous day OHLCV metrics
for x in data.columns[2:-6]:
  data[f'Growth Rate_{x}'] = (data[x] - data[x].shift(1))/data[x].shift(1) * 100
data["PDO"] = data["Open"].shift(1)
data["PDH"] = data["High"].shift(1)
data["PDL"] = data["Low"].shift(1)
data["PDC"] = data["Close"].shift(1)
data["PDV"] = data["Volume"].shift(1)

# For row in data, if date is first trading date, set above column values to null. Note: date is the index:
data.loc[first_trading_date, ['PDO', 'PDH', 'PDL', 'PDC', 'PDV', 'Growth Rate_Open', 'Growth Rate_High', 'Growth Rate_Low',
    'Growth Rate_Close']] = np.nan

data

,Date_col,Ticker,Open,High,Low,Close,Volume,rank,Close_Open_Diff,Close_Open_Diff_Perc,...,Close_Open_Diff_Ternary,Growth Rate_Open,Growth Rate_High,Growth Rate_Low,Growth Rate_Close,PDO,PDH,PDL,PDC,PDV
0,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0,-0.292272,-0.252231,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0,1.100878,0.953745,...,2.0,-0.386754,0.191521,0.819737,0.817596,115.875028,116.995388,114.092184,115.582756,2030700.0
2,2021-01-06,A,115.631477,120.687730,115.163842,119.723244,2428500.0,3.0,4.091767,3.538627,...,2.0,0.177254,2.958785,0.118583,2.742255,115.426878,117.219459,115.027439,116.527756,2344000.0
3,2021-01-07,A,121.126152,124.194981,120.911820,122.908997,1775900.0,4.0,1.782845,1.471891,...,2.0,4.751885,2.906054,4.991130,2.660931,115.631477,120.687730,115.163842,119.723244,2428500.0
4,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0,0.555311,0.450628,...,1.0,1.737295,0.125493,1.401967,0.713361,121.126152,124.194981,120.911820,122.908997,1775900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480386,2024-12-26,ZTS,163.226540,165.290293,162.528653,165.021118,2167200.0,1002.0,1.794578,1.099440,...,2.0,0.012220,0.454435,0.172053,0.497879,163.206597,164.542554,162.249499,164.203583,1023600.0
480387,2024-12-27,ZTS,164.452833,166.028073,163.027150,164.103897,1800100.0,1003.0,-0.348936,-0.212180,...,1.0,0.751283,0.446354,0.306713,-0.555820,163.226540,165.290293,162.528653,165.021118,2167200.0
480388,2024-12-30,ZTS,163.396028,163.555549,160.963380,161.751007,1531400.0,1004.0,-1.645021,-1.006769,...,0.0,-0.642619,-1.489220,-1.265906,-1.433781,164.452833,166.028073,163.027150,164.103897,1800100.0
480389,2024-12-31,ZTS,162.409011,163.256455,161.382116,162.438919,1327400.0,1005.0,0.029908,0.018415,...,1.0,-0.604064,-0.182870,0.260144,0.425291,163.396028,163.555549,160.963380,161.751007,1531400.0


Getting gap (i.e., change from some previous day metric) features and volume features

Derived Features:
* Gap up/gap down: (Open - PDC) / PDC; Could be either binary (e.g., |X| > 2%) or metric
* Daily range gap up: (Open - PDH) / PDH; Must be positive; if 0 or negative, then open price has not gapped up from previous day range
* Daily range gap down: (Open - PDL) / PDL; Must be negative; if 0 or positive, then open price has not gapped down from previous day range
* Average volume (14 days)

In [25]:
# Getting the opening gap features from PDC
data["Gap_from_PDC"] = (data["Open"] - data["PDC"]) / data["PDC"]
gap_from_PDC_threshold = 0.02  # 2%
data["Gap_Binary_Threshold"] = abs(data["Gap_from_PDC"]) > gap_from_PDC_threshold

# Getting daily range gap up features
data["Daily_Range_Gap_Up"] = (data["Open"] - data["PDH"]) / data["PDH"]
data["Daily_Range_Gap_Up"] = data["Daily_Range_Gap_Up"].clip(lower = 0)
data["DR_Gap_Up_Binary"] = data["Daily_Range_Gap_Up"] > 0

# Getting daily range gap down features
data["Daily_Range_Gap_Down"] = (data["Open"] - data["PDL"]) / data["PDL"]
data["Daily_Range_Gap_Down"] = data["Daily_Range_Gap_Down"].clip(upper = 0)
data["DR_Gap_Down_Binary"] = data["Daily_Range_Gap_Down"] < 0

# Getting 14-day average volume and previous volume features
avg_vol_window = 14
data["Avg_Volume_14D"] = data["Volume"].rolling(window=avg_vol_window).mean()
data["Avg_Vol_Prev_14D"] = data["PDV"].rolling(window=avg_vol_window).mean()

# Comparing current volume to average volume over the past XX days
data["Vol_Change"] = (data["Volume"] - data["Avg_Vol_Prev_14D"]) / data["Avg_Vol_Prev_14D"]

# Setting the first 14-day volume averages and previous averages of every ticker to np.nan
data.loc[data["rank"] < 15, ["Avg_Volume_14D", "Avg_Vol_Prev_14D", "Vol_Change"]] = np.nan

data[10:20]

,Date_col,Ticker,Open,High,Low,Close,Volume,rank,Close_Open_Diff,Close_Open_Diff_Perc,...,Growth Rate_Close_Open_Diff_Perc,Gap_from_PDC,Gap_Binary_Threshold,Daily_Range_Gap_Up,DR_Gap_Up_Binary,Daily_Range_Gap_Down,DR_Gap_Down_Binary,Avg_Volume_14D,Avg_Vol_Prev_14D,Vol_Change
10,2021-01-19,A,124.097524,124.506707,123.035616,124.068298,1677200.0,11.0,-0.029226,-0.023551,...,-110.985193,0.009270,False,0.002045,True,0.000000,False,NaN,NaN,NaN
11,2021-01-20,A,124.419037,125.597856,124.078058,124.808731,1843000.0,12.0,0.389694,0.313211,...,-1429.947197,0.002827,False,0.000000,False,0.000000,False,NaN,NaN,NaN
12,2021-01-21,A,124.477488,125.461463,123.162277,123.727333,1612100.0,13.0,-0.750155,-0.602643,...,-292.408034,-0.002654,False,0.000000,False,0.000000,False,NaN,NaN,NaN
13,2021-01-22,A,123.805266,124.506713,123.045367,123.074593,1438700.0,14.0,-0.730673,-0.590179,...,-2.068118,0.000630,False,0.000000,False,0.000000,False,NaN,NaN,NaN
14,2021-01-25,A,123.425317,124.194960,121.223553,122.246498,1365400.0,15.0,-1.178819,-0.955086,...,61.829855,0.002850,False,0.000000,False,0.000000,False,1.728993e+06,1.776514e+06,-0.231416
15,2021-01-26,A,122.265987,122.704388,120.677993,120.726700,1476600.0,16.0,-1.539287,-1.258966,...,31.816922,0.000159,False,0.000000,False,0.000000,False,1.667036e+06,1.728993e+06,-0.145977
16,2021-01-27,A,118.934124,119.830415,115.066426,115.738647,1923500.0,17.0,-3.195477,-2.686762,...,113.410258,-0.014848,False,0.000000,False,-0.014451,True,1.630964e+06,1.667036e+06,0.153845
17,2021-01-28,A,115.933492,119.508918,115.719160,118.427521,1677900.0,18.0,2.494029,2.151258,...,-180.068821,0.001683,False,0.000000,False,0.000000,False,1.623964e+06,1.630964e+06,0.028778
18,2021-01-29,A,118.086535,118.554169,116.206269,117.073334,1822000.0,19.0,-1.013201,-0.858016,...,-139.884365,-0.002879,False,0.000000,False,0.000000,False,1.641450e+06,1.623964e+06,0.121946
19,2021-02-01,A,117.579932,118.943849,117.083072,117.716324,1292700.0,20.0,0.136392,0.115999,...,-113.519467,0.004327,False,0.000000,False,0.000000,False,1.609014e+06,1.641450e+06,-0.212465


Getting technical and momentum indicators

Derived Features:

* SMA (simple moving average) (14-day, 50-day, 200-day): average of the last XX closing prices
* EMA (exponential moving average) (9-day, 20-day):
(Opentoday * Smoothing1 + Days) + EMAyesterday * (1 - Smoothing1 + Days)
Smoothing: usually starts with a factor of 2
Increasing the smoothing will put more emphasis on today’s open
Initial calculations for EMA yesterday can be calculated from SMA
https://www.investopedia.com/terms/e/ema.asp
https://www.strike.money/technical-analysis/ema
* RSI (relative strength index) (14-day):
“A momentum indicator used in technical analysis. RSI measures the speed and magnitude of a security’s recent price changes to detect overbought or oversold conditions in the price of that security.”
https://www.investopedia.com/terms/r/rsi.asp
RSI = 100 - 1001 + (Previous Average Gain * 13) + Current Gain(Previous Average Loss * 13) + Current Loss
Previous average gain: average of the past 13 days where the close was greater than the open
Previous average loss: average of the past 13 days where the close was lower than the open
* ATR (average true range) (14-day):
Shows the average range of the past XX days
(1/n) * Σpast 14 days (PDC - PDL)
This is a simplified version of ATR that we can use as either a binary or metric indicator
Likely needs to be standardized (an ATR of 1 dollar might be a lot for a 10 dollar stock, but might be nothing for a 300 dollar stock)
* Can also consider ATR% (ATRP): (ATR / PDC) * 100
https://www.investopedia.com/terms/a/atr.asp


In [28]:
# SMA and EMA including this period
data["SMA_14"] = data["Close"].rolling(window=14).mean()
data["SMA_50"] = data["Close"].rolling(window=50).mean()
data["SMA_200"] = data["Close"].rolling(window=200).mean()
data["EMA_9"] = data["Close"].ewm(span=9, adjust=False).mean()
data["EMA_20"] = data["Close"].ewm(span=20, adjust=False).mean()

# SMA and EMA for the past XXX periods
data["SMA_prev_14"] = data["PDC"].rolling(window=14).mean()
data["SMA_prev_50"] = data["PDC"].rolling(window=50).mean()
data["SMA_prev_200"] = data["PDC"].rolling(window=200).mean()
data["EMA_prev_9"] = data["PDC"].ewm(span=9, adjust=False).mean()
data["EMA_prev_20"] = data["PDC"].ewm(span=20, adjust=False).mean()

# RSI Calculation
delta = data["Close"].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
data["RSI_14"] = 100 - (100 / (1 + rs))

# RSI for the previous XX periods using tulipy
rsi_prev_14 = ti.rsi(data["PDC"].to_numpy(), 14)
nan_array_rsi = np.full(14, np.nan)
rsi_prev_14 = np.concatenate((nan_array_rsi, rsi_prev_14))
data["RSI_prev_14"] = rsi_prev_14

# ATR Calculation
data["ATR_14"] = (data["PDH"] - data["PDL"]).rolling(window=14).mean()
data["NATR_14"] = (data["ATR_14"] / data["PDC"]) * 100

# Setting 9, 14, 20, 50, and 200 day metrics to np.nan:
data.loc[data["rank"] <= 9, ['EMA_9', 'EMA_prev_9']] = np.nan
data.loc[data["rank"] <= 14, ['SMA_14', 'SMA_prev_14', 'RSI_14', 'RSI_prev_14', 'ATR_14', 'NATR_14']] = np.nan
data.loc[data["rank"] <= 20, ['EMA_20', 'EMA_prev_20']] = np.nan
data.loc[data["rank"] <= 50, ['SMA_50', 'SMA_prev_50']] = np.nan
data.loc[data["rank"] <= 200, ['SMA_200', 'SMA_prev_200']] = np.nan

data[:18]

,Date_col,Ticker,Open,High,Low,Close,Volume,rank,Close_Open_Diff,Close_Open_Diff_Perc,...,EMA_20,SMA_prev_14,SMA_prev_50,SMA_prev_200,EMA_prev_9,EMA_prev_20,RSI_14,RSI_prev_14,ATR_14,NATR_14
0,2021-01-04,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0,-0.292272,-0.252231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-05,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0,1.100878,0.953745,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-06,A,115.631477,120.687730,115.163842,119.723244,2428500.0,3.0,4.091767,3.538627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-07,A,121.126152,124.194981,120.911820,122.908997,1775900.0,4.0,1.782845,1.471891,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-08,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0,0.555311,0.450628,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2021-01-11,A,123.191497,126.007024,122.977173,124.847687,1746800.0,6.0,1.656190,1.344403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2021-01-12,A,125.023060,125.081511,122.947943,123.698105,1826400.0,7.0,-1.324955,-1.059768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2021-01-13,A,124.175467,124.750267,123.474019,123.707840,1530400.0,8.0,-0.467627,-0.376585,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2021-01-14,A,123.990362,124.886660,122.577731,122.782318,924900.0,9.0,-1.208044,-0.974305,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2021-01-15,A,122.694648,123.844241,121.525567,122.957687,2115400.0,10.0,0.263039,0.214385,...,NaN,NaN,NaN,NaN,121.719017,NaN,NaN,NaN,NaN,NaN


Getting sectors for stocks

In [33]:
# Getting the sector for stocks as another feature
sp500_tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
ticker_and_sector = sp500_tables[0][['Symbol', 'GICS Sector']]

# Getting the GICS sector into the table through a merge function
data = data.reset_index().merge(ticker_and_sector, left_on='Ticker', right_on='Symbol', how='left').set_index('Date_col')
data = data.drop(columns=['Symbol'])
data

,index,Ticker,Open,High,Low,Close,Volume,rank,Close_Open_Diff,Close_Open_Diff_Perc,...,SMA_prev_14,SMA_prev_50,SMA_prev_200,EMA_prev_9,EMA_prev_20,RSI_14,RSI_prev_14,ATR_14,NATR_14,GICS Sector
Date_col,,,,,,,,,,,,,,,,,,,,,
2021-01-04,0,A,115.875028,116.995388,114.092184,115.582756,2030700.0,1.0,-0.292272,-0.252231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health Care
2021-01-05,1,A,115.426878,117.219459,115.027439,116.527756,2344000.0,2.0,1.100878,0.953745,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health Care
2021-01-06,2,A,115.631477,120.687730,115.163842,119.723244,2428500.0,3.0,4.091767,3.538627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health Care
2021-01-07,3,A,121.126152,124.194981,120.911820,122.908997,1775900.0,4.0,1.782845,1.471891,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health Care
2021-01-08,4,A,123.230471,124.350838,122.606963,123.785782,1577200.0,5.0,0.555311,0.450628,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Health Care
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-26,480386,ZTS,163.226540,165.290293,162.528653,165.021118,2167200.0,1002.0,1.794578,1.099440,...,171.711605,177.056477,175.363787,167.838326,171.386876,31.784154,31.300147,3.671758,2.236101,Health Care
2024-12-27,480387,ZTS,164.452833,166.028073,163.027150,164.103897,1800100.0,1003.0,-0.348936,-0.212180,...,171.052882,176.529766,175.291711,167.274884,170.780613,25.913882,33.659957,3.668909,2.223296,Health Care
2024-12-30,480388,ZTS,163.396028,163.555549,160.963380,161.751007,1531400.0,1004.0,-1.645021,-1.006769,...,170.208292,175.943930,175.240379,166.640687,170.144735,18.544864,32.318655,3.659651,2.230081,Health Care


In [34]:
# Getting date index column:
date_index = data.index.sort_values()

# Dropping duplicates:
date_index = date_index.drop_duplicates()

date_index

DatetimeIndex(['2021-01-04', '2021-01-05', '2021-01-06', '2021-01-07',
               '2021-01-08', '2021-01-11', '2021-01-12', '2021-01-13',
               '2021-01-14', '2021-01-15',
               ...
               '2024-12-18', '2024-12-19', '2024-12-20', '2024-12-23',
               '2024-12-24', '2024-12-26', '2024-12-27', '2024-12-30',
               '2024-12-31',        'NaT'],
              dtype='datetime64[ns]', name='Date_col', length=1006, freq=None)

In [35]:
# Turning data into CSV:
data.to_csv('data/engineered_data_baseline.csv')

In [76]:
!tar -czvf data/engineered_data_baseline.csv.tar.gz data/engineered_data_baseline.csv

data/engineered_data_baseline.csv


In [89]:
# !git lfs track "*.tar.gz"
# !git add .gitattributes
# !git add data/engineered_data_baseline.csv.tar.gz
# !git commit -m "Add compressed CSV file only"
# !git push origin arun-branch

"*.tar.gz" already supported
fatal: cannot exec '.git/hooks/post-commit': Permission denied
[arun-branch cffec31] Add compressed CSV file only
 2 files changed, 5 insertions(+)
 create mode 100644 .gitattributes
 create mode 100644 data/engineered_data_baseline.csv.tar.gz
Uploading LFS objects: 100% (1/1), 126 MB | 0 B/s, done.
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 594 bytes | 18.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/aagarwal17/datasci-207-final-project.git
   358013e..cffec31  arun-branch -> arun-branch
